# 01 · Dataset, Policy Knowledge Base & Retrieval Tool

This notebook introduces the three ingredients of the complaint-resolution RAG system:

1. the **bank complaints dataset** (the inputs + ground-truth labels),
2. the **policy knowledge base** (the documents the agent grounds answers in), and
3. the **retrieval tool** that finds the right policy for a complaint.

> Note: complaint narratives are pre-processed (lemmatized, stop-words removed), so they
> read as keyword sequences. Classification and retrieval still work well on this signal.

## 1. The bank complaints dataset

In [ ]:
from aieng.agent_evals.complaint_resolution.data import BankComplaintsDataset


dataset = BankComplaintsDataset()
print("Total examples:", len(dataset))
print("Categories:", dataset.get_categories())

In [ ]:
example = dataset[0]
print("category:", example.category)
print("narrative:", example.narrative[:200], "...")

A **class-balanced** sample (the raw data is ~56% `credit_reporting`):

In [ ]:
from collections import Counter


balanced = dataset.sample_balanced(n_per_category=3, random_state=42)
print(Counter(e.category for e in balanced))

## 2. The policy knowledge base\n\nFive hand-authored policy documents, one per category.

In [ ]:
from aieng.agent_evals.complaint_resolution.kb import CATEGORY_TO_POLICY_ID, PolicyKnowledgeBase


kb = PolicyKnowledgeBase()
for doc in kb.docs:
    print(f"{doc.id:22s} {doc.category:20s} {doc.title}")

print()
print("category -> gold policy id:")
for cat, pid in CATEGORY_TO_POLICY_ID.items():
    print(f"  {cat:20s} -> {pid}")

## 3. The retrieval tool

`retrieve_policy` embeds the complaint and returns the most similar policy document(s) by
cosine similarity. This is the **R** in RAG — the agent calls it before writing a resolution.

In [ ]:
# The KB retrieve() is async — notebooks support top-level await.
query = "charged twice same purchase credit card duplicate transaction unauthorized"
results = await kb.retrieve(query, k=2)
for doc in results:
    print(f"{doc.id:22s} score={doc.score:.3f}  {doc.title}")

The top hit should be `POL-CREDIT-CARD`. Try the other categories:

In [ ]:
queries = {
    "debt_collection": "collection agency calling debt already paid harassment",
    "mortgages_and_loans": "mortgage escrow payment misapplied wrong amount servicing",
    "credit_reporting": "incorrect late payment credit report not mine dispute",
    "retail_banking": "overdraft fee error checking account deposit hold",
}
for cat, q in queries.items():
    top = (await kb.retrieve(q, k=1))[0]
    print(f"{cat:20s} -> {top.id} (gold: {CATEGORY_TO_POLICY_ID[cat]})")

await kb.aclose()